# Tensorflow

In [1]:
import numpy as np
import tensorflow as tf

x_tf = tf.constant([1.0,0.5])
print(x_tf)
# shape=(2,) means 1 DIMENSION holding 2 values.
# There is no second dimension here at all; this is a flat vector,
# not a row or column vector (those only exist once you're 2D:
# [[1.0, 0.5]] = 1x2 row vector, [[1.0],[0.5]] = 2x1 column vector)


w_tf = tf.constant([[1,2,3],[4,5,6]]) # shape(2, 3)
print(w_tf.shape)
# double brackets = nested lists = 2 dimensions. Outer list = rows,
# each inner list = one row's values. shape=(2,3): 2 rows, 3 columns —
# same rows-by-columns idea as df.shape, but tensors aren't locked to
# 2D the way a DataFrame always is

# element-wise ops: matching POSITIONS combine
a_tf = tf.constant([1.0, 2.0, 3.0])
b_tf = tf.constant([4.0, 5.0, 6.0])
print(a_tf + b_tf)
print(a_tf * b_tf)

# Matrix multiplication
v_tf = tf.constant([[1, 0], [0, 1], [1, 1]])  # shape (3, 2)
print(tf.matmul(w_tf, v_tf))
# THIS is the real mechanism behind a neuron's weighted sum:
# X (1xN row vector of inputs) matmul W (NxM matrix of weights,
# one COLUMN per neuron) = one raw z per neuron, computed all at
# once instead of looping w1*x1+w2*x2+... for each neuron separately

tf.Tensor([1.  0.5], shape=(2,), dtype=float32)
(2, 3)
tf.Tensor([5. 7. 9.], shape=(3,), dtype=float32)
tf.Tensor([ 4. 10. 18.], shape=(3,), dtype=float32)
tf.Tensor(
[[ 4  5]
 [10 11]], shape=(2, 2), dtype=int32)


## Defining a layer

In [2]:
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import Sequential


# Reset all relevant TF/Keras RNGs each rerun of this cell
keras.utils.set_random_seed(42)
tf.keras.backend.clear_session()

first_hidden_layer_tf = layers.Dense(units=2, activation='sigmoid')
# a Dense layer creates its weights automatically the FIRST TIME data
# actually flows through it (this is called the layer "building"),
# not at the moment layers.Dense(...) is written

x_input_tf = tf.constant([[1.0, 0.5]])
_ = first_hidden_layer_tf(x_input_tf)  # forces the layer to create its weights (values don't matter here)
# calling this ONCE first is required before set_weights() can work —
# the layer has ZERO weights until it's been "built" by seeing real
# input at least once

# Manually fix the weights
my_weights = np.array([[0.4, -0.3], [0.6, 0.8]], dtype=np.float32) # row0=[0.4,-0.3] is x1's weight to each neuron; row1=[0.6,0.8] is x2's
my_biases = np.array([0.1, 0.2], dtype=np.float32)
first_hidden_layer_tf.set_weights([my_weights, my_biases])

output = first_hidden_layer_tf(x_input_tf)
print(output)

# Second hidden layer
second_hidden_layer_tf = layers.Dense(units=2, activation='sigmoid')
# no input_shape here either — automatically infers 2 inputs, since
# that's how many outputs first_hidden_layer produces (units=2)

model_tf = Sequential([
    first_hidden_layer_tf,
    second_hidden_layer_tf
])
# Sequential = "run these layers in this exact order, one layer's
# output automatically becomes the next layer's input"

model_output = model_tf(x_input_tf)
print(model_output)



tf.Tensor([[0.6899745  0.57444257]], shape=(1, 2), dtype=float32)
tf.Tensor([[0.7657091  0.57721543]], shape=(1, 2), dtype=float32)


# Pytorch

In [3]:
import torch

x_torch = torch.tensor([1.0, 0.5])
print(x_torch)

w_torch = torch.tensor([[1,2,3],[4,5,6]]) # shape (2, 3)
print(w_torch.shape)

# element-wise ops: matching POSITIONS combine
a_torch = torch.tensor([1.0, 2.0, 3.0])
b_torch = torch.tensor([4.0, 5.0, 6.0])
print(a_torch + b_torch)
print(a_torch * b_torch)

# Matrix multiplication
v_torch = torch.tensor([[1, 0], [0, 1], [1, 1]])  # shape (3, 2)
print(torch.matmul(w_torch, v_torch))


tensor([1.0000, 0.5000])
torch.Size([2, 3])
tensor([5., 7., 9.])
tensor([ 4., 10., 18.])
tensor([[ 4,  5],
        [10, 11]])


## Defining a layer

In [4]:
import random
import torch.nn as nn
# Reset Python/NumPy/Torch RNGs each rerun of this cell
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

first_hidden_layer_torch =nn.Linear(in_features=2, out_features=2)
# UNLIKE Keras's Dense: weights are created IMMEDIATELY here
print(first_hidden_layer_torch.weight)
print(first_hidden_layer_torch.bias)

x_input_torch = torch.tensor([[1.0,0.5]])
output = first_hidden_layer_torch(x_input_torch)
print(output)

activated_output = torch.sigmoid(output)
print(activated_output)

# Second hidden layer
second_hidden_layer_torch =nn.Linear(in_features=2, out_features=2)
model_torch = nn.Sequential(
    first_hidden_layer_torch,
    nn.Sigmoid(),
    second_hidden_layer_torch,
    nn.Sigmoid()
)
output_torch = model_torch(x_input_torch)
print(output_torch)

Parameter containing:
tensor([[ 0.5406,  0.5869],
        [-0.1657,  0.6496]], requires_grad=True)
Parameter containing:
tensor([-0.1549,  0.1427], requires_grad=True)
tensor([[0.6791, 0.3018]], grad_fn=<AddmmBackward0>)
tensor([[0.6635, 0.5749]], grad_fn=<SigmoidBackward0>)
tensor([[0.6513, 0.5616]], grad_fn=<SigmoidBackward0>)


## Loss Functions and Optimizers

In [ ]:
output_layer_torch = nn.Linear(in_features=2, out_features =1)
full_model_torch = nn.Sequential(
    first_hidden_layer_torch,
    nn.Sigmoid(),
    second_hidden_layer_torch,
    nn.Sigmoid(),
    output_layer_torch,
    nn.Sigmoid()
)

# One iteration only to see how each step works
loss_fn_torch = torch.nn.BCELoss()

optimizer_torch = torch.optim.Adam(full_model_torch.parameters(), lr=0.01)

true_label_torch = torch.tensor([[1.0]]) # pretend the correct answer was "1"

prediction_torch = full_model_torch(x_input_torch)

loss_torch = loss_fn_torch(prediction_torch, true_label_torch)

loss_torch.backward()      # compute gradients for EVERY weight, based on this forward pass
optimizer_torch.step()     # apply the update to EVERY weight, using those gradients
optimizer_torch.zero_grad()  # clear gradients, ready for the next round


# loop for optimizing weights
# COMPLETE TRAINING CYCLE, one epoch = these 5 steps, repeated:
# 1. predict (forward pass, using CURRENT weights)
# 2. compute loss (compare prediction vs true label)
# 3. backward() — compute gradient for EVERY weight/bias at once,
#    based on THIS forward pass (accumulates on top of old grads
#    if not cleared — this is exactly why zero_grad() matters)
# 4. step() — apply new_weight = old_weight - lr*gradient,
#    to EVERY weight simultaneously (not one at a time)
# 5. zero_grad() — clear grads so NEXT backward() starts clean,
#    rather than adding on top of this iteration's leftover values
for epoch in range(50):
    prediction_torch = full_model_torch(x_input_torch)
    loss_torch = loss_fn_torch(prediction_torch, true_label_torch)

    loss_torch.backward()
    optimizer_torch.step()
    optimizer_torch.zero_grad()

    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss:{loss_torch.item()}")
print(f"Final loss: {loss_torch.item()}")
print(f"Final output: {prediction_torch}")



Epoch 0, Loss:0.5754730701446533
Epoch 10, Loss:0.4594395160675049
Epoch 20, Loss:0.36038386821746826
Epoch 30, Loss:0.280208557844162
Epoch 40, Loss:0.21846236288547516
Final loss: 0.17663006484508514
Final output: tensor([[0.8381]], grad_fn=<SigmoidBackward0>)
